# PhishARG — Entrenamiento del Modelo Híbrido NLP con GPU

**Proyecto de Tesis:** PhishARG — Detección de Phishing con IA Explicable

**Autores:** Gabriel Adia & Tomás Basualdo | **Universidad:** UADE

Este notebook entrena el clasificador híbrido **Sentence Transformers (MiniLM) + XGBoost** aprovechando la GPU T4 gratuita de Google Colab.

> **Antes de empezar:** En el menú superior, andá a `Entorno de ejecución` > `Cambiar tipo de entorno de ejecución` > seleccioná **T4 GPU** > **Guardar**.


## Paso 1: Clonar el repositorio y cambiar a la rama experimental

In [ ]:
# Clonar el repositorio de PhishARG y posicionarse en la rama hibrida
!git clone https://github.com/gabrieladia1979/flujo_v2.git
%cd flujo_v2
!git checkout codex/hybrid-nlp
print("\nRepositorio clonado y rama codex/hybrid-nlp activa.")

## Paso 2: Instalar dependencias

In [ ]:
# Instalar las librerias necesarias para el modelo hibrido
!pip install -q sentence-transformers==5.7.0 xgboost scikit-learn

# Verificar que CUDA esta disponible
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print("ADVERTENCIA: No hay GPU. Activa la GPU en Entorno de ejecucion > Cambiar tipo.")

## Paso 3: Subir el dataset de entrenamiento

El archivo `multilingual-v1.csv` (19 MB) está en tu PC en:
`flujo_v2\artifacts\hybrid\corpus\multilingual-v1.csv`

Ejecutá esta celda y seleccioná ese archivo cuando te lo pida.

In [ ]:
import shutil
from google.colab import files

# Crear la estructura de carpetas
!mkdir -p artifacts/hybrid/corpus
!mkdir -p reports

print("Selecciona el archivo multilingual-v1.csv desde tu PC:")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, "artifacts/hybrid/corpus/" + filename)
    size_mb = len(uploaded[filename]) / (1024 * 1024)
    print(f"Archivo {filename} subido correctamente ({size_mb:.1f} MB)")

## Paso 4: Entrenar el modelo con GPU

Configuración potenciada respecto al entrenamiento local:
- **3 épocas** (antes era 1) — el modelo ve los datos 3 veces
- **256 tokens** (antes era 128) — lee el doble de contexto por correo
- **4 capas entrenables** (antes eran 2) — ajusta más parámetros de la red neuronal
- **batch-size 32** (antes era 16) — aprovecha la memoria de la GPU
- **`--device cuda`** — usa la GPU en lugar de la CPU

> Tiempo estimado: **~2 minutos** en T4 GPU (vs 40 min en CPU).

In [ ]:
# Entrenar el modelo hibrido con la potencia de la GPU
!python scripts/train_hybrid.py \
  --dataset artifacts/hybrid/corpus/multilingual-v1.csv \
  --phishing-label 1 \
  --output artifacts/hybrid/multilingual-candidate-colab \
  --report reports/hybrid_multilingual_colab \
  --epochs 3 \
  --batch-size 32 \
  --max-tokens 256 \
  --trainable-layers 4 \
  --device cuda

## Paso 5: Evaluar el modelo y comparar con el modelo original

Probamos el nuevo modelo contra los 14 casos difíciles del corpus de evaluación y comparamos con el modelo legacy (TF-IDF + XGBoost puro).

In [ ]:
# Evaluar el modelo hibrido contra los 14 edge-cases y comparar con el legacy
!python scripts/evaluate_hybrid.py \
  --model artifacts/hybrid/multilingual-candidate-colab \
  --dataset data/classifier_eval_v1.jsonl \
  --output reports/hybrid_eval_colab \
  --compare-legacy

## Paso 6: Ver los resultados del entrenamiento

Leemos el reporte generado para ver las métricas de cada candidato.

In [ ]:
import json
from pathlib import Path

# Leer el reporte de entrenamiento
report = json.loads(Path("reports/hybrid_multilingual_colab.json").read_text())

print("=" * 70)
print("RESULTADOS DEL ENTRENAMIENTO HIBRIDO (Google Colab + GPU)")
print("=" * 70)

for name, r in report["results"].items():
    m = r["test_metrics"]
    cm = m["confusion_matrix"]
    print(f"  {name}")
    print(f"    F1-Score:  {m['f1']:.4f}")
    print(f"    Precision: {m['precision']:.4f}")
    print(f"    Recall:    {m['recall']:.4f}")
    print(f"    FP: {cm['false_positive']}  |  FN: {cm['false_negative']}")
    print()

print("=" * 70)

## Paso 7: Descargar el modelo entrenado a tu PC

Descargá el ZIP y descomprimilo en tu repositorio local dentro de:
`flujo_v2\artifacts\hybrid\multilingual-candidate-colab\`

In [ ]:
import shutil
from google.colab import files

# Comprimir modelo y reportes
shutil.make_archive("multilingual-candidate-colab", "zip",
                    "artifacts/hybrid/multilingual-candidate-colab")
shutil.make_archive("reports_colab", "zip", "reports")

print("Archivos comprimidos. Descargando...")

# Descargar automaticamente
files.download("multilingual-candidate-colab.zip")
files.download("reports_colab.zip")

print("\nDescarga completa!")
print("Descomprimilo en: flujo_v2/artifacts/hybrid/multilingual-candidate-colab/")
print("Para probarlo en tu PC:")
print("  $env:PHISHARG_HYBRID_MODEL_DIR = 'artifacts/hybrid/multilingual-candidate-colab'")
print("  .venv-hybrid/Scripts/python.exe -m uvicorn main:app --port 8001")

## (Opcional) Entrenamiento avanzado

Si querés exprimir aún más el modelo, descomentá y corré esta celda con **5 épocas y 384 tokens**.
Solo tarda ~5 minutos más en la T4.

In [ ]:
# OPCIONAL: Entrenamiento todavia mas agresivo (5 epocas, 384 tokens)
# Descomentar las lineas de abajo para correrlo:

# !python scripts/train_hybrid.py \
#   --dataset artifacts/hybrid/corpus/multilingual-v1.csv \
#   --phishing-label 1 \
#   --output artifacts/hybrid/multilingual-candidate-colab-v2 \
#   --report reports/hybrid_multilingual_colab_v2 \
#   --epochs 5 \
#   --batch-size 32 \
#   --max-tokens 384 \
#   --trainable-layers 4 \
#   --device cuda